# Isolation Forest Baseline — обнаружение аномалий в умном доме

**Дисциплина:** Прикладной искусственный интеллект для систем умного дома  
**Модуль:** M3 (Разработка моделей МО)  
**Компетенция КРМ:** **ML-4** — Обучение без учителя (unsupervised learning)

## Цель
Построить baseline-модель обнаружения аномалий в сенсорных данных (температура, влажность, движение, освещённость) с помощью Isolation Forest.

## Что получим
- Обученную модель + scaler
- Метрики precision / recall / F1
- Отчёт и предсказания


In [ ]:
# %pip install scikit-learn pandas numpy joblib matplotlib -q

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import joblib
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)
print("Библиотеки загружены")


## 1. Генерация / загрузка данных

In [ ]:
# Генерируем синтетический датасет (если файла ещё нет)
DATA_PATH = Path("../data/smart_home_sensors.csv")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    N = 2000
    timestamps = pd.date_range("2026-01-01", periods=N, freq="5min")
    hour = (timestamps.hour + timestamps.minute / 60.0).to_numpy(dtype=float)

    temperature = 21 + 2 * np.sin(2 * np.pi * hour / 24) + RNG.normal(0, 0.4, N)
    humidity = 45 + 8 * np.sin(2 * np.pi * (hour - 6) / 24) + RNG.normal(0, 2, N)
    motion = (RNG.random(N) < 0.12).astype(float)
    lux = np.clip(300 * np.sin(np.pi * hour / 24) ** 2 + RNG.normal(0, 30, N), 0, 800)

    # Аномалии ~5%
    n_anom = int(0.05 * N)
    anom_idx = RNG.choice(N, size=n_anom, replace=False)
    for i, idx in enumerate(anom_idx):
        kind = i % 4
        if kind == 0:
            temperature[idx] += RNG.choice([-8, 10])
        elif kind == 1:
            humidity[idx] = RNG.uniform(85, 98)
        elif kind == 2 and 1 <= hour[idx] <= 4:
            motion[idx] = 1.0
            lux[idx] = RNG.uniform(0, 15)
        else:
            lux[idx] = RNG.uniform(900, 1200)

    df = pd.DataFrame({
        "timestamp": timestamps,
        "temperature": np.round(temperature, 2),
        "humidity": np.round(humidity, 1),
        "motion": motion.astype(int),
        "lux": np.round(lux, 1),
        "is_anomaly": 0,
    })
    df.loc[anom_idx, "is_anomaly"] = 1
    df.to_csv(DATA_PATH, index=False)
    print(f"Создан датасет: {DATA_PATH} ({len(df)} строк)")
else:
    df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
    print(f"Загружен датасет: {DATA_PATH} ({len(df)} строк)")

df["hour"] = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60.0
df.head()


## 2. Быстрый EDA

In [ ]:
print("Распределение аномалий:")
print(df["is_anomaly"].value_counts())
print("\nСтатистики признаков:")
display(df[["temperature", "humidity", "motion", "lux"]].describe())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["temperature", "humidity", "motion", "lux"]):
    ax.hist(df[col], bins=40, alpha=0.7, color="steelblue")
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 3. Обучение Isolation Forest

In [ ]:
FEATURES = ["temperature", "humidity", "motion", "lux", "hour"]
X = df[FEATURES].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples="auto",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_scaled)

scores = model.decision_function(X_scaled)   # чем меньше — тем аномальнее
preds = (model.predict(X_scaled) == -1).astype(int)

df["pred_anomaly"] = preds
df["anomaly_score"] = scores
print("Модель обучена")


## 4. Оценка качества

In [ ]:
y_true = df["is_anomaly"].values
y_pred = df["pred_anomaly"].values

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print()
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))


## 5. Сохранение артефактов

In [ ]:
MODEL_DIR = Path("../models")
REPORT_DIR = Path("../reports")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, MODEL_DIR / "isolation_forest.joblib")
joblib.dump(scaler, MODEL_DIR / "scaler.joblib")
df.to_csv(REPORT_DIR / "predictions.csv", index=False)

print("Сохранено:")
print(f"  - {MODEL_DIR / 'isolation_forest.joblib'}")
print(f"  - {MODEL_DIR / 'scaler.joblib'}")
print(f"  - {REPORT_DIR / 'predictions.csv'}")


## Выводы и следующие шаги

Isolation Forest — простой и эффективный baseline для unsupervised anomaly detection в IoT-данных умного дома.

**Что улучшить (для более высоких баллов):**
1. Добавить временные признаки (rolling mean/std, lag).
2. Сравнить с LOF, One-Class SVM, Autoencoder.
3. Оптимизировать contamination / threshold по precision-recall кривой.
4. Экспортировать модель в ONNX / TFLite для Edge-развёртывания (модуль M4).
5. Интегрировать с MQTT + Home Assistant (публикация событий аномалий).

**Компетенции:** ML-4 (unsupervised), ML-3 (анализ ошибок), LC-5 (полный цикл).
